# Train Attention U-Net trên Google Colab
**Đề tài:** Phân đoạn tổn thương trên phim X-quang nha khoa

### Thứ tự chạy:
1. Đổi Runtime sang GPU: `Runtime > Change runtime type > T4 GPU`
2. Chạy từng cell theo thứ tự từ trên xuống

### Yêu cầu chuẩn bị trước (làm 1 lần duy nhất trên máy tính):
- Upload thư mục `data/processed/` (đã có train/val/test) lên Google Drive
- Đường dẫn khuyến nghị: `My Drive/DO_AN_NHA_KHOA/data/processed/`

## Bước 1 — Kiểm tra GPU

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU sẵn sàng : {gpu_name}')
    print(f'VRAM         : {gpu_mem:.1f} GB')
else:
    print('CẢNH BÁO: Không tìm thấy GPU!')
    print('Vào Runtime > Change runtime type > chọn T4 GPU rồi chạy lại.')

## Bước 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Kiểm tra thư mục data trên Drive có tồn tại không
DRIVE_DATA_PATH = '/content/drive/MyDrive/DO_AN_NHA_KHOA/data/processed'

if os.path.exists(DRIVE_DATA_PATH):
    splits = ['train', 'val', 'test']
    for s in splits:
        n_imgs  = len(os.listdir(os.path.join(DRIVE_DATA_PATH, s, 'images')))
        n_masks = len(os.listdir(os.path.join(DRIVE_DATA_PATH, s, 'masks')))
        print(f'{s:5s}: {n_imgs} ảnh | {n_masks} mask')
    print('\nDrive mount thành công!')
else:
    print(f'CẢNH BÁO: Không tìm thấy data tại {DRIVE_DATA_PATH}')
    print('Hãy upload thư mục data/processed/ lên Drive trước.')

## Bước 3 — Clone code từ GitHub

In [ ]:
# ==============================================================
#  ĐỔI ĐƯỜNG DẪN GITHUB CỦA NHÓM VÀO ĐÂY
# ==============================================================
GITHUB_URL  = 'https://github.com/TEN_NHOM/DO_AN_NHA_KHOA.git'
GITHUB_BRANCH = 'main'   # hoặc 'master' tùy repo
PROJECT_DIR = '/content/DO_AN_NHA_KHOA'
# ==============================================================

if os.path.exists(PROJECT_DIR):
    # Nếu đã clone rồi thì chỉ pull code mới nhất
    print('Repo đã tồn tại, đang pull code mới nhất...')
    %cd {PROJECT_DIR}
    !git pull origin {GITHUB_BRANCH}
else:
    # Clone lần đầu
    print('Đang clone repo...')
    !git clone -b {GITHUB_BRANCH} {GITHUB_URL} {PROJECT_DIR}
    %cd {PROJECT_DIR}

print(f'\nThư mục làm việc: {os.getcwd()}')
!ls -la

## Bước 4 — Cài thư viện

In [ ]:
# Colab đã có sẵn torch, chỉ cần cài thêm opencv
!pip install -q opencv-python scikit-learn

# Kiểm tra version
import torch, cv2, sklearn
print(f'PyTorch  : {torch.__version__}')
print(f'OpenCV   : {cv2.__version__}')
print(f'sklearn  : {sklearn.__version__}')
print('Cài đặt hoàn tất!')

## Bước 5 — Link dữ liệu từ Drive vào project

> Dùng symlink thay vì copy để tiết kiệm thời gian và dung lượng.

In [ ]:
DATA_TARGET = os.path.join(PROJECT_DIR, 'data', 'processed')

# Tạo thư mục data/ nếu chưa có
os.makedirs(os.path.join(PROJECT_DIR, 'data'), exist_ok=True)

# Tạo symlink (nếu symlink đã tồn tại thì xoá rồi tạo lại)
if os.path.exists(DATA_TARGET) or os.path.islink(DATA_TARGET):
    os.remove(DATA_TARGET) if os.path.islink(DATA_TARGET) else None
    print('Symlink cũ đã được xoá.')

os.symlink(DRIVE_DATA_PATH, DATA_TARGET)
print(f'Symlink tạo thành công:')
print(f'  {DATA_TARGET}')
print(f'  -> {DRIVE_DATA_PATH}')

# Kiểm tra lại
for split in ['train', 'val', 'test']:
    n = len(os.listdir(os.path.join(DATA_TARGET, split, 'images')))
    print(f'  {split}: {n} ảnh')

## Bước 6 — Kiểm tra nhanh model trước khi train

In [ ]:
import sys
sys.path.insert(0, PROJECT_DIR)

from models.attention_unet import AttentionUNet
from scripts.dataset import DentalDataset
from torch.utils.data import DataLoader

# Kiểm tra model
model = AttentionUNet(n_classes=1)
dummy = torch.randn(1, 1, 256, 256)
out   = model(dummy)
assert out.shape == (1, 1, 256, 256)
total_params = sum(p.numel() for p in model.parameters())
print(f'Model OK — output shape: {out.shape}')
print(f'Tổng tham số: {total_params:,}')

# Kiểm tra DataLoader
ds     = DentalDataset(DATA_TARGET, split='train')
loader = DataLoader(ds, batch_size=4)
imgs, masks = next(iter(loader))
print(f'DataLoader OK — imgs: {imgs.shape}, masks: {masks.shape}')
print('\nSẵn sàng train!')

## Bước 7 — Tạo thư mục lưu model

In [ ]:
# Lưu models_saved vào Drive để không mất khi Colab disconnect
DRIVE_MODELS_PATH = '/content/drive/MyDrive/DO_AN_NHA_KHOA/models_saved'
os.makedirs(DRIVE_MODELS_PATH, exist_ok=True)

# Symlink models_saved/ trong project về Drive
LOCAL_MODELS = os.path.join(PROJECT_DIR, 'models_saved')
if os.path.islink(LOCAL_MODELS):
    os.remove(LOCAL_MODELS)
elif os.path.isdir(LOCAL_MODELS):
    import shutil
    shutil.rmtree(LOCAL_MODELS)

os.symlink(DRIVE_MODELS_PATH, LOCAL_MODELS)
print(f'Model sẽ được lưu vào Drive: {DRIVE_MODELS_PATH}')

## Bước 8 — TRAIN MÔ HÌNH

Chạy cell nào tương ứng với mô hình muốn train.

In [ ]:
# ── Attention U-Net ────────────────────────────────────────────
%cd {PROJECT_DIR}
!python train.py \
    --arch attention_unet \
    --save_name attention_unet_best.pth \
    --epochs 50 \
    --batch_size 8 \
    --lr 1e-4

In [ ]:
# ── VGG-UNet (chạy sau khi có file vgg_unet.py) ────────────────
# %cd {PROJECT_DIR}
# !python train.py \
#     --arch vgg_unet \
#     --save_name vgg_unet_best.pth \
#     --epochs 50 \
#     --batch_size 8 \
#     --lr 1e-4

In [ ]:
# ── ResNet-UNet (chạy sau khi có file res_unet.py) ─────────────
# %cd {PROJECT_DIR}
# !python train.py \
#     --arch res_unet \
#     --save_name res_unet_best.pth \
#     --epochs 50 \
#     --batch_size 8 \
#     --lr 1e-4

## Bước 9 — Đánh giá sau khi train xong

In [ ]:
%cd {PROJECT_DIR}
!python evaluate.py \
    --model_path models_saved/attention_unet_best.pth \
    --output_dir results/attention_unet \
    --num_samples 10

## Bước 10 — Xem kết quả trực quan

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

results_dir = os.path.join(PROJECT_DIR, 'results', 'attention_unet')
samples = sorted([f for f in os.listdir(results_dir) if f.startswith('sample_')])

fig, axes = plt.subplots(min(5, len(samples)), 1, figsize=(22, 5 * min(5, len(samples))))
if len(samples) == 1:
    axes = [axes]

for ax, fname in zip(axes, samples[:5]):
    img = mpimg.imread(os.path.join(results_dir, fname))
    ax.imshow(img)
    ax.set_title(fname, fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.show()

# In báo cáo
report = os.path.join(results_dir, 'metrics_report.txt')
print('\n' + open(report, encoding='utf-8').read())